## **preprocessing**

In [ ]:
!pip install scanpy
!pip install --upgrade requests
!pip install commot
!pip uninstall numpy
!pip install "numpy<2.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 78.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Would remove:
    /usr/local/bin/f2py
    /usr/local/bin/numpy-config
    /usr/local/lib/python3.12/dist-packages/numpy-2.0.2.dist-info/*
    /usr/local/lib/python3.12/dist-packages/numpy.libs/libgfortran-040039e1-0352e75f.so.5.0.0
    /usr/local/lib/python3.12/dist-packages/numpy.libs/libquadmath-96973f99-934c22de.so.0.0.0
    /usr/local/lib/python3.12/dist-packages/numpy.libs/libscipy_openblas64_-99b71e71.so
    /usr/local/lib/python3.12/dist-packages/numpy/*
Proceed (Y/n)? y
  Successfully uninstalled numpy-2.0.2
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scanpy 1.12.1 requires numpy>=2, bu

In [ ]:
import scanpy as sc
import commot as ct
import numpy as np
import anndata
from sklearn.preprocessing import StandardScaler
import pandas as pd

adata = sc.read_h5ad("lymph_node_50k_sample.h5ad")

tile_size = 1700
x_min, y_min = 4.26, 3.46
x_max, y_max = 3637.84, 7228.18

coords = adata.obsm['spatial']

signaling_matrix = None

for x_start in np.arange(x_min, x_max, tile_size):
    for y_start in np.arange(y_min, y_max, tile_size):
        x_end = x_start + tile_size
        y_end = y_start + tile_size

        # Mask
        mask = (
            (coords[:, 0] >= x_start) & (coords[:, 0] < x_end) &
            (coords[:, 1] >= y_start) & (coords[:, 1] < y_end)
        )

        if mask.sum() < 50:  # skip tiles with too few cells
            continue

        if mask.sum() > 28000:
          idx = np.random.choice(np.where(mask)[0], 30000, replace=False)
          mask = np.zeros(len(coords), dtype=bool)
          mask[idx] = True

        print(f"Processing tile ({x_start:.0f}-{x_end:.0f}, {y_start:.0f}-{y_end:.0f}) — {mask.sum()} cells")

        adata_sub = adata[mask].copy()
        sc.pp.normalize_total(adata_sub, inplace=True)
        sc.pp.log1p(adata_sub)

        df_ligrec = ct.pp.ligand_receptor_database(database='CellChat', species='human')
        genes_in_panel = set(adata_sub.var_names)
        lr_mask = df_ligrec.apply(
            lambda row: row[0] in genes_in_panel and
                        all(g in genes_in_panel for g in row[1].split('_')), axis=1
        )
        df_ligrec = df_ligrec[lr_mask].reset_index(drop=True)
        print(f"{len(df_ligrec)} LR pairs found in panel")

        # COMMOT
        ct.tl.spatial_communication(
            adata_sub,
            database_name='user_database',
            df_ligrec=df_ligrec,
            dis_thr=50,
            heteromeric=True
        )


        # CCC
        ccc_sender = adata_sub.obsm['commot-user_database-sum-sender']
        ccc_receiver = adata_sub.obsm['commot-user_database-sum-receiver']
        ccc_features = ccc_sender.values + ccc_receiver.values
        print(ccc_features)


        if signaling_matrix is None:
            n_features = ccc_features.shape[1]
            signaling_matrix = np.zeros((adata.n_obs, n_features))

        print(ccc_features)
        cell_indices = np.where(mask)[0]
        n_feat = min(ccc_features.shape[1], signaling_matrix.shape[1])
        signaling_matrix[cell_indices, :] = ccc_features[:, :]


        # Free memory
        del adata_sub
        import gc
        gc.collect()



adata.obsm['signaling'] = signaling_matrix
print("Done. Signaling matrix shape:", signaling_matrix.shape)
adata.write_h5ad("lymph_node_50k_with_signaling.h5ad")

Processing tile (4-1704, 3-1703) — 225 cells
337 LR pairs found in panel
[[0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   8.75]
 [0.   0.   0.   ... 0.   0.   0.01]
 ...
 [0.   0.   0.   ... 0.   0.   2.28]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.02]]
[[0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   8.75]
 [0.   0.   0.   ... 0.   0.   0.01]
 ...
 [0.   0.   0.   ... 0.   0.   2.28]
 [0.   0.   0.   ... 0.   0.   0.  ]
 [0.   0.   0.   ... 0.   0.   0.02]]
Processing tile (4-1704, 1703-3403) — 1997 cells
337 LR pairs found in panel
[[0.   0.   0.   ... 0.   0.   2.83]
 [1.28 0.   0.   ... 0.   0.   1.54]
 [0.   0.   0.   ... 0.   0.   6.54]
 ...
 [0.   0.   0.   ... 0.   0.   2.96]
 [0.   0.   0.   ... 0.   0.   3.48]
 [0.   0.   0.   ... 0.   0.   0.  ]]
[[0.   0.   0.   ... 0.   0.   2.83]
 [1.28 0.   0.   ... 0.   0.   1.54]
 [0.   0.   0.   ... 0.   0.   6.54]
 ...
 [0.   0.   0.   ... 0.   0.   2.96]
 [0.   0.   0